In [1]:
# ─────────────────────────────────────────────
# 준비 — 라이브러리 불러오기
# ─────────────────────────────────────────────
import numpy as np
import pandas as pd
from pathlib import Path


# ─────────────────────────────────────────────
# 모두마켓 이번 달 주문 데이터 생성 (가상)
#   → 일부러 결측·이상치·표기 혼재를 섞어 둠
# ─────────────────────────────────────────────
np.random.seed(42)   # 매번 같은 결과가 나오도록 고정
n = 1500

# --- 범주형 값: p = 각 값이 뽑힐 확률 ---
regions    = np.random.choice(["서울", "경기", "부산", "인천", "대구"], n,
                              p=[0.4, 0.25, 0.15, 0.1, 0.1])
membership = np.random.choice(["basic", "silver", "gold", "vip"], n,
                              p=[0.5, 0.25, 0.15, 0.1])
channels   = np.random.choice(["web", "app", "app ", "APP"], n,      # 표기 혼재 포함
                              p=[0.5, 0.4, 0.05, 0.05])
categories = np.random.choice(["패션", "뷰티", "식품", "가전", "도서"], n)

# --- 가격·수량·금액 ---
prices = np.random.choice([9900, 19900, 29900, 49900, 89900, 129900, 249900], n,
                          p=[0.2, 0.25, 0.2, 0.15, 0.1, 0.06, 0.04])
quantities = np.random.choice([1, 1, 1, 2, 2, 3], n)   # 1개가 가장 많게
amount = (prices * quantities).astype(float)           # 금액 = 가격 × 수량

orders = pd.DataFrame({
    "order_id":     [f"O{str(i).zfill(5)}" for i in range(1, n + 1)],   # O00001 형식
    "customer_age": np.random.normal(35, 9, n).round().astype(int),     # 평균 35세
    "region":       regions,
    "membership":   membership,
    "channel":      channels,
    "category":     categories,
    "price":        prices.astype(float),
    "quantity":     quantities,
    "amount":       amount,
})

# --- 오염 심기: 결측 · 이상치 · 표기 혼재 ---
orders.loc[np.random.choice(n, 60, replace=False), "amount"] = np.nan        # 결측 60건
orders.loc[np.random.choice(n, 30, replace=False), "customer_age"] = np.nan  # 결측 30건
orders.loc[5,  "customer_age"] = 999      # 입력 실수 이상치
orders.loc[8,  "customer_age"] = -3       # 불가능한 음수
orders.loc[12, "quantity"]     = 80       # 비정상적으로 큰 주문
orders.loc[20, "region"]       = " 서울 " # 앞뒤 공백
orders.loc[21, "region"]       = "Seoul"  # 영문 표기
orders.loc[40, "membership"]   = "VIP"    # 대소문자 혼재

print("이번 달 주문 데이터 준비 완료:", orders.shape)   # (1500, 9)
orders.head()

이번 달 주문 데이터 준비 완료: (1500, 9)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount
0,O00001,30.0,서울,silver,app,패션,19900.0,1,NaN
1,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0
2,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0
3,O00004,24.0,경기,basic,app,뷰티,19900.0,1,NaN
4,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0


In [2]:
# ─────────────────────────────────────────────
# 시나리오 0 — 원본 CSV 파일 준비
#   실무에서는 외부에서 CSV를 받아오는 단계.
#   여기서는 만든 orders를 CSV로 저장해서 "받아온 원본 파일"처럼 사용
# ─────────────────────────────────────────────

# --- 작업 폴더 & 파일 경로 ---
work_dir = Path("d006_work")
work_dir.mkdir(exist_ok=True)              # 폴더가 이미 있어도 에러 안 남

input_csv  = work_dir / "orders_raw.csv"   # 원본 (전처리 전)
output_csv = work_dir / "orders_clean.csv" # 결과 (전처리 후)

# --- 원본 저장 ---
orders.to_csv(input_csv, index=False)      # index=False: 행 번호는 저장 안 함

print("원본 CSV 저장:", input_csv)
print("파일 크기:", input_csv.stat().st_size, "bytes")

원본 CSV 저장: d006_work\orders_raw.csv
파일 크기: 81901 bytes


In [3]:
# ─────────────────────────────────────────────
# 시나리오 1 — 전처리 단계 함수 6개
#   함수 하나 = 단계 하나. 모든 함수는 df를 받아 새 df를 돌려줌
#   (원본을 직접 고치지 않음 → 다음 셀에서 .pipe()로 이어 붙이기 쉬움)
# ─────────────────────────────────────────────

# ① 불러오기
def load_orders(path):
    """CSV에서 주문 데이터를 읽어 옵니다."""
    return pd.read_csv(path)


# ② 문자열 정리
def clean_strings_full(df):
    """문자열 컬럼의 공백·대소문자·표기 혼재를 정리합니다."""
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),  # " 서울 ", "Seoul" → "서울"
        membership=df["membership"].str.lower(),                     # "VIP" → "vip"
        channel=df["channel"].str.strip().str.lower(),               # "app ", "APP" → "app"
    )


# ③ 잘못된 행 제거
def drop_invalid_rows(df, age_min=10, age_max=80, qty_max=10):
    """결측과 불가능한 값을 가진 행을 제거합니다."""
    return (
        df
        .dropna(subset=["amount", "customer_age"])          # 결측 행 제거
        .query("@age_min < customer_age < @age_max")       # 나이 999, -3 제거
        .query("quantity <= @qty_max")                     # 수량 80 제거
        .drop_duplicates()                                 # 중복 행 제거
        .reset_index(drop=True)                            # 행 번호 0부터 다시
    )


# ④ 파생 컬럼 추가
def add_derived(df):
    """분석용 파생 컬럼을 추가합니다."""
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),                          # 금액 로그 (치우침 완화)
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),  # 우수 회원 여부 1/0
        amount_class=lambda d: np.where(d["amount"] >= 100_000, "high",      # 10만 원 이상
                               np.where(d["amount"] >= 30_000,  "mid",       # 3만 원 이상
                                                                "low")),     # 그 외
    )


# ⑤ 인코딩
def encode_full(df):
    """membership은 순서형(Ordinal), 나머지 범주형은 One-Hot으로 인코딩합니다."""
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}   # 등급 순서 그대로 숫자로
    out = df.assign(membership_ord=df["membership"].map(order_map))

    one_hots = [
        pd.get_dummies(out["region"],   prefix="region", dtype=int),
        pd.get_dummies(out["channel"],  prefix="ch",     dtype=int),
        pd.get_dummies(out["category"], prefix="cat",    dtype=int),
    ]
    return pd.concat([out] + one_hots, axis=1)   # 옆으로(axis=1) 붙이기


# ⑥ 스케일링 (sklearn 없이 직접 계산)
def scale_with_robust(df, cols=("customer_age", "amount", "quantity")):
    """수치형 컬럼을 Robust 방식으로 스케일링하고 _scaled 컬럼을 추가합니다.
    Robust 스케일링 = (값 - 중앙값) / IQR   ← sklearn RobustScaler와 같은 계산
    """
    cols = list(cols)
    median = df[cols].median()
    iqr = df[cols].quantile(0.75) - df[cols].quantile(0.25)
    scaled_df = ((df[cols] - median) / iqr).add_suffix("_scaled")
    return pd.concat([df, scaled_df], axis=1)


print("단계 함수 6개 정의 완료. 다음 셀에서 한 줄로 조립합니다.")

단계 함수 6개 정의 완료. 다음 셀에서 한 줄로 조립합니다.


In [4]:
# ─────────────────────────────────────────────
# 시나리오 2 — end-to-end 파이프라인
#   단계 함수 6개를 .pipe()로 순서대로 이어 붙임
#   → 원본 CSV 경로만 넣으면 전처리 완료된 표가 나옴
# ─────────────────────────────────────────────
def preprocess(input_path):
    """원본 CSV 경로를 받아 전처리된 DataFrame을 반환합니다."""
    return (
        load_orders(input_path)                                         # ① 불러오기
        .pipe(clean_strings_full)                                       # ② 문자열 정리
        .pipe(drop_invalid_rows, age_min=10, age_max=80, qty_max=10)    # ③ 잘못된 행 제거
        .pipe(add_derived)                                              # ④ 파생 컬럼
        .pipe(encode_full)                                              # ⑤ 인코딩
        .pipe(scale_with_robust)                                        # ⑥ 스케일링
    )


# --- 한 줄 호출 ---
clean_df = preprocess(input_csv)

print("입력 행 수 → 출력 행 수:", orders.shape[0], "→", clean_df.shape[0])   # 1500 → 1404
print("입력 컬럼 수 → 출력 컬럼 수:", orders.shape[1], "→", clean_df.shape[1]) # 9 → 28
clean_df.head(3)

입력 행 수 → 출력 행 수: 1500 → 1404
입력 컬럼 수 → 출력 컬럼 수: 9 → 28


,order_id,customer_age,region,membership,channel,category,price,quantity,amount,amount_log,...,ch_app,ch_web,cat_가전,cat_도서,cat_뷰티,cat_식품,cat_패션,customer_age_scaled,amount_scaled,quantity_scaled
0,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0,13.527430,...,1,0,1,0,0,0,0,-0.916667,10.141429,2.0
1,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0,9.898525,...,0,1,0,0,0,0,1,0.833333,-0.284286,0.0
2,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0,9.898525,...,1,0,0,0,0,0,1,0.500000,-0.284286,0.0


In [5]:
# ─────────────────────────────────────────────
# 시나리오 3 — 저장 + 재로드 검증
#   저장한 파일을 다시 읽어서, 저장 전과 똑같은지 확인
#   (저장 과정에서 행·열이 빠지거나 index가 끼어들지 않았는지 점검)
# ─────────────────────────────────────────────

# --- 저장 ---
clean_df.to_csv(output_csv, index=False)
print("저장 파일:", output_csv)
print("파일 크기:", output_csv.stat().st_size, "bytes")

# --- 다시 읽기 ---
reloaded = pd.read_csv(output_csv)

# --- 검증 ---
same_shape = clean_df.shape == reloaded.shape
same_cols  = list(clean_df.columns) == list(reloaded.columns)

print("\n저장 직전 shape:", clean_df.shape)    # (1404, 28)
print("다시 읽은 shape:", reloaded.shape)
print("shape 동일?   :", same_shape)
print("컬럼 이름 동일?:", same_cols)

저장 파일: d006_work\orders_clean.csv
파일 크기: 199683 bytes

저장 직전 shape: (1404, 28)
다시 읽은 shape: (1404, 28)
shape 동일?   : True
컬럼 이름 동일?: True


In [6]:
# ─────────────────────────────────────────────
# 시나리오 4 — 품질 리포트
#   quality_report()       : 전처리 전·후 지표를 계산해서 dict로 반환
#   print_quality_report() : 그 dict를 보기 좋게 출력
# ─────────────────────────────────────────────
def quality_report(before: pd.DataFrame, after: pd.DataFrame, top_n: int = 5) -> dict:
    """전처리 전·후 데이터의 품질 지표를 계산해 dict로 반환합니다."""
    # 1) 행 수 변화
    rows_before, rows_after = len(before), len(after)
    removed_pct = round(100 * (1 - rows_after / rows_before), 2) if rows_before else 0.0

    # 2) 결측 비율(%) 상위 N개 — 결측이 없는 컬럼은 제외
    def missing_top(df):
        pct = df.isna().mean().mul(100).round(2)
        return pct[pct > 0].sort_values(ascending=False).head(top_n).to_dict()

    # 3) 새로 생긴 컬럼을 종류별로 분류
    new_cols = [c for c in after.columns if c not in before.columns]
    encoded = [c for c in new_cols
               if c.startswith(("region_", "ch_", "cat_")) or c.endswith("_ord")]
    scaled  = [c for c in new_cols if c.endswith("_scaled")]
    derived = [c for c in new_cols if c not in encoded + scaled]

    return {
        "rows_before": rows_before,
        "rows_after": rows_after,
        "removed_pct": removed_pct,
        "cols_before": before.shape[1],
        "cols_after": after.shape[1],
        "new_cols": {"derived": derived, "encoded": encoded, "scaled": scaled},
        "missing_top_before": missing_top(before),
        "missing_top_after": missing_top(after),
        "dtypes_before": before.dtypes.astype(str).value_counts().to_dict(),
        "dtypes_after": after.dtypes.astype(str).value_counts().to_dict(),
    }


def print_quality_report(rep: dict) -> None:
    """quality_report()가 만든 dict를 보기 좋게 출력합니다."""
    line = "─" * 48
    print(line)
    print(" 📊 데이터 품질 리포트")
    print(line)

    # [1] 행 수
    removed = rep["rows_before"] - rep["rows_after"]
    print("\n[1] 행 수 변화")
    print(f"   입력 {rep['rows_before']:>6,}행  →  출력 {rep['rows_after']:>6,}행")
    print(f"   제거: {removed:,}행 ({rep['removed_pct']:.2f}%)")

    # [2] 결측 비율
    print("\n[2] 컬럼별 결측 비율 (상위 5개)")
    print(f"   {'컬럼':<16}{'전처리 전':>10}{'전처리 후':>10}")
    before_m, after_m = rep["missing_top_before"], rep["missing_top_after"]
    if not before_m and not after_m:
        print("   결측 없음 ✅")
    for col in dict.fromkeys(list(before_m) + list(after_m)):   # 순서 유지 + 중복 제거
        b = f"{before_m[col]:.2f}%" if col in before_m else "-"
        a = f"{after_m[col]:.2f}%" if col in after_m else "0.00%"
        print(f"   {col:<16}{b:>10}{a:>10}")
    if before_m and not after_m:
        print("   → 전처리 후 결측 0개 ✅")

    # [3] 새 컬럼
    nc = rep["new_cols"]
    total_new = sum(len(v) for v in nc.values())
    print(f"\n[3] 새로 만들어진 컬럼: {total_new}개  "
          f"(컬럼 수 {rep['cols_before']} → {rep['cols_after']})")
    labels = {"derived": "파생", "encoded": "인코딩", "scaled": "스케일링"}
    for key, cols in nc.items():
        print(f"   {labels[key]:<6} {len(cols):>2}개: {', '.join(cols) if cols else '-'}")

    # [4] 자료형
    print("\n[4] 자료형 분포 (전처리 전 → 후)")
    all_types = dict.fromkeys(list(rep["dtypes_before"]) + list(rep["dtypes_after"]))
    for t in all_types:
        b = rep["dtypes_before"].get(t, 0)
        a = rep["dtypes_after"].get(t, 0)
        print(f"   {t:<16}{b:>3}개 → {a:>3}개")
    print(line)


# --- 사용: 원본(orders) vs 전처리 결과(clean_df) ---
rep = quality_report(orders, clean_df)
print_quality_report(rep)

────────────────────────────────────────────────
 📊 데이터 품질 리포트
────────────────────────────────────────────────

[1] 행 수 변화
   입력  1,500행  →  출력  1,404행
   제거: 96행 (6.40%)

[2] 컬럼별 결측 비율 (상위 5개)
   컬럼                   전처리 전     전처리 후
   amount               4.00%     0.00%
   customer_age         2.00%     0.00%
   → 전처리 후 결측 0개 ✅

[3] 새로 만들어진 컬럼: 19개  (컬럼 수 9 → 28)
   파생      3개: amount_log, is_premium, amount_class
   인코딩    13개: membership_ord, region_경기, region_대구, region_부산, region_서울, region_인천, ch_app, ch_web, cat_가전, cat_도서, cat_뷰티, cat_식품, cat_패션
   스케일링    3개: customer_age_scaled, amount_scaled, quantity_scaled

[4] 자료형 분포 (전처리 전 → 후)
   str               5개 →   6개
   float64           3개 →   7개
   int64             1개 →  15개
────────────────────────────────────────────────
